In [1]:
from conch.open_clip_custom import create_model_from_pretrained, tokenize, get_tokenizer
import torch
import os
from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
# remember 'conda activate conch'


/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [8]:
model_cfg       = 'conch_ViT-B-16'
checkpoint_paths = ['checkpoints/conch/pytorch_model.bin', 
                    'checkpoints/conch/conch_mhist_finetuned_num_unfrozen=1.bin',
                    'checkpoints/conch/conch_mhist_finetuned_num_unfrozen=2.bin',
                    'checkpoints/conch/conch_mhist_finetuned_num_unfrozen=4.bin']
image_dir       = 'mhist_data/images'
annotations_dir = 'mhist_data/annotations.csv'
device          = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE      = 64

In [9]:
# ── Prompt definitions (shared across all models) ─────────────────────────────
classnames = {
    "HP": [
        "hyperplastic polyp",
        "hyperplastic colorectal polyp",
        "colonic hyperplastic polyp with elongated crypts",
        "benign serrated polyp with superficial luminal serration",
        "hyperplastic polyp with straight crypt bases and goblet cells",
    ],
    "SSA": [
        "sessile serrated adenoma",
        "sessile serrated lesion",
        "sessile serrated adenoma with broad-based dilated crypts",
        "serrated polyp with basal crypt dilation and L-shaped crypts",
        "sessile serrated lesion with horizontal crypt growth and heavy serration",
    ],
}

templates = [
    "CLASSNAME.",
    "a photomicrograph showing CLASSNAME.",
    "a photomicrograph of CLASSNAME.",
    "a histopathological image showing CLASSNAME.",
    "a histopathological image of CLASSNAME.",
    "a histopathological photograph of CLASSNAME.",
    "a histopathological photograph showing CLASSNAME.",
    "shows CLASSNAME.",
    "presence of CLASSNAME.",
    "CLASSNAME is present.",
    "an H&E stained image of CLASSNAME.",
    "an H&E stained image showing CLASSNAME.",
    "an H&E image showing CLASSNAME.",
    "an H&E image of CLASSNAME.",
    "CLASSNAME, H&E stain.",
    "CLASSNAME, H&E.",
]

# ── Helpers ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def build_zeroshot_classifier(model, classnames, templates, tokenizer, device):
    class_embeddings = []
    class_order = list(classnames.keys())
    for cls in class_order:
        synonyms = classnames[cls]
        prompts = [t.replace("CLASSNAME", s) for s in synonyms for t in templates]
        tokenized = tokenize(texts=prompts, tokenizer=tokenizer).to(device)
        text_features = model.encode_text(tokenized)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        class_embedding = text_features.mean(dim=0)
        class_embedding = class_embedding / class_embedding.norm()
        class_embeddings.append(class_embedding)
    weights = torch.stack(class_embeddings, dim=1).to(device)
    return weights, class_order


class MHISTInferenceDataset(torch.utils.data.Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.image_dir, row['Image Name'])).convert('RGB')
        return self.transform(image), row['Majority Vote Label']


@torch.no_grad()
def run_zero_shot(model, loader, zeroshot_weights, class_order, device):
    preds, gts, scores = [], [], []
    for images, labels in tqdm(loader, desc="Zero-shot inference"):
        images = images.to(device)
        image_features = model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        logits = image_features @ zeroshot_weights
        batch_preds = logits.argmax(dim=1).cpu().tolist()
        preds.extend([class_order[p] for p in batch_preds])
        gts.extend(labels)
        scores.append(logits.cpu())
    return preds, gts, torch.cat(scores)

In [10]:
# ── Shared resources (loaded once) ────────────────────────────────────────────
df = pd.read_csv(annotations_dir)
tokenizer = get_tokenizer()

# ── Loop over checkpoints ─────────────────────────────────────────────────────
summary_rows = []
per_image_frames = []

for ckpt in checkpoint_paths:
    print(f"\n{'='*72}\nEvaluating checkpoint: {ckpt}\n{'='*72}")

    # Load this checkpoint
    model, preprocess = create_model_from_pretrained(model_cfg, ckpt, device=device)
    model = model.eval()

    # Re-encode prompts (text encoder weights may differ across checkpoints)
    zeroshot_weights, class_order = build_zeroshot_classifier(
        model, classnames, templates, tokenizer, device
    )

    # Dataset/loader (preprocess could in principle vary, so rebuild)
    dataset = MHISTInferenceDataset(df, image_dir, preprocess)
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
    )

    # Inference
    preds, gts, all_scores = run_zero_shot(model, loader, zeroshot_weights, class_order, device)

    # Metrics
    acc     = accuracy_score(gts, preds)
    bal_acc = balanced_accuracy_score(gts, preds)
    print(f"\nAccuracy:          {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print("\nClassification report:")
    print(classification_report(gts, preds, digits=4))

    summary_rows.append({
        'checkpoint_path':   ckpt,
        'accuracy':          acc,
        'balanced_accuracy': bal_acc,
    })

    # Per-image predictions for this checkpoint (optional but handy)
    per_image_frames.append(pd.DataFrame({
        'checkpoint_path': ckpt,
        'image':           df['Image Name'].values,
        'gt':              gts,
        'pred':            preds,
        'score_HP':        all_scores[:, class_order.index('HP')].tolist(),
        'score_SSA':       all_scores[:, class_order.index('SSA')].tolist(),
    }))

    # Free GPU memory before loading the next checkpoint
    del model, zeroshot_weights, loader, dataset
    torch.cuda.empty_cache()


Evaluating checkpoint: checkpoints/conch/pytorch_model.bin


Zero-shot inference:   0%|          | 0/50 [00:00<?, ?it/s]/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Zero-shot inference: 100%|██████████| 50/50 [15:23<00:00, 18.47s/it]



Accuracy:          0.6904
Balanced Accuracy: 0.5279

Classification report:
              precision    recall  f1-score   support

          HP     0.6986    0.9648    0.8104      2162
         SSA     0.5422    0.0909    0.1557       990

    accuracy                         0.6904      3152
   macro avg     0.6204    0.5279    0.4831      3152
weighted avg     0.6495    0.6904    0.6048      3152


Evaluating checkpoint: checkpoints/conch/conch_mhist_finetuned_num_unfrozen=1.bin


Zero-shot inference:   0%|          | 0/50 [00:00<?, ?it/s]/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Zero-shot inference: 100%|██████████| 50/50 [15:24<00:00, 18.48s/it]



Accuracy:          0.7240
Balanced Accuracy: 0.6107

Classification report:
              precision    recall  f1-score   support

          HP     0.7423    0.9154    0.8198      2162
         SSA     0.6235    0.3061    0.4106       990

    accuracy                         0.7240      3152
   macro avg     0.6829    0.6107    0.6152      3152
weighted avg     0.7050    0.7240    0.6913      3152


Evaluating checkpoint: checkpoints/conch/conch_mhist_finetuned_num_unfrozen=2.bin


Zero-shot inference:   0%|          | 0/50 [00:00<?, ?it/s]/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Zero-shot inference: 100%|██████████| 50/50 [15:10<00:00, 18.21s/it]



Accuracy:          0.7497
Balanced Accuracy: 0.6590

Classification report:
              precision    recall  f1-score   support

          HP     0.7712    0.9029    0.8319      2162
         SSA     0.6618    0.4152    0.5102       990

    accuracy                         0.7497      3152
   macro avg     0.7165    0.6590    0.6711      3152
weighted avg     0.7369    0.7497    0.7309      3152


Evaluating checkpoint: checkpoints/conch/conch_mhist_finetuned_num_unfrozen=4.bin


Zero-shot inference:   0%|          | 0/50 [00:00<?, ?it/s]/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Zero-shot inference: 100%|██████████| 50/50 [14:57<00:00, 17.94s/it]



Accuracy:          0.7817
Balanced Accuracy: 0.7089

Classification report:
              precision    recall  f1-score   support

          HP     0.8023    0.9047    0.8504      2162
         SSA     0.7115    0.5131    0.5962       990

    accuracy                         0.7817      3152
   macro avg     0.7569    0.7089    0.7233      3152
weighted avg     0.7738    0.7817    0.7706      3152



In [11]:
# ── Summary ───────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(summary_rows)
print("\n\n" + "="*72)
print("MODEL COMPARISON")
print("="*72)
print(results_df.to_string(index=False))

results_df.to_csv('mhist_zeroshot_model_comparison.csv', index=False)
print("\nSaved per-model accuracies to mhist_zeroshot_model_comparison.csv")

# Also save per-image predictions across all models (long format, keyed by checkpoint_path)
per_image_df = pd.concat(per_image_frames, ignore_index=True)
per_image_df.to_csv('mhist_zeroshot_per_image_all_models.csv', index=False)
print("Saved per-image predictions across all models to mhist_zeroshot_per_image_all_models.csv")



MODEL COMPARISON
                                           checkpoint_path  accuracy  balanced_accuracy
                       checkpoints/conch/pytorch_model.bin  0.690355           0.527878
checkpoints/conch/conch_mhist_finetuned_num_unfrozen=1.bin  0.723985           0.610708
checkpoints/conch/conch_mhist_finetuned_num_unfrozen=2.bin  0.749683           0.659010
checkpoints/conch/conch_mhist_finetuned_num_unfrozen=4.bin  0.781726           0.708925

Saved per-model accuracies to mhist_zeroshot_model_comparison.csv
Saved per-image predictions across all models to mhist_zeroshot_per_image_all_models.csv
